# Transformer (Pre-LN) (최적화 v2 - SentencePiece 적용)
## 영어→한국어 기계번역 | Korean-English TED Talks

### v2 핵심 변경: 토크나이저 전면 교체
- **기존**: Mecab 형태소 분석 (한국어) + 공백 분리 (영어)
  - 문제: 과도한 분절, OOV 다수, 어휘 크기 ~35K/~39K
- **변경**: **SentencePiece BPE** (영어 16K + 한국어 16K)
  - OOV 문제 해소 (서브워드로 분해)
  - 어휘 크기 대폭 감소 → 학습 효율 향상
  - 영어/한국어 모두 데이터 기반 토큰화로 일관성 확보


# 1. 환경 설정

In [1]:
!pip install datasets sentencepiece -q
!pip install evaluate rouge_score sacrebleu -q

In [19]:
import re
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import math
import sentencepiece as spm
from collections import Counter
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")
print(f"사용 가능한 GPU 수: {torch.cuda.device_count()}")

사용 디바이스: cuda:0
사용 가능한 GPU 수: 4


---
# 2. SentencePiece 토크나이저 로드

### 사전 학습 필요
```bash
# 먼저 별도 스크립트로 SentencePiece 모델을 학습해야 합니다:
python train_sentencepiece.py
```

학습이 완료되면 `en_spm.model`, `ko_spm.model` 파일이 생성됩니다.

In [20]:

# ============================================================
# 데이터 로드
# ============================================================
dataset = load_dataset("msarmi9/korean-english-multitarget-ted-talks-task")
num_samples = 88000

en_sentences = []
ko_sentences = []

for i, data in enumerate(dataset['train']):
    if i >= num_samples:
        break
    en_sentences.append(data['english'].strip())
    ko_sentences.append(data['korean'].strip())

print(f"영어 문장 수: {len(en_sentences)}")
print(f"한국어 문장 수: {len(ko_sentences)}")

영어 문장 수: 88000
한국어 문장 수: 88000


In [21]:
# ============================================================
# 사전 학습된 SentencePiece 모델 로드
# (python train_sentencepiece.py 로 미리 학습 필요)
# ============================================================
sp_en = spm.SentencePieceProcessor()
sp_ko = spm.SentencePieceProcessor()
sp_en.load('en_spm.model')
sp_ko.load('ko_spm.model')

src_vocab_size = sp_en.get_piece_size()
tar_vocab_size = sp_ko.get_piece_size()

print(f"영어 어휘 크기: {src_vocab_size}")
print(f"한국어 어휘 크기: {tar_vocab_size}")

# 토큰화 테스트
test_en = "Have you had dinner?"
test_ko = "저녁은 드셨나요?"

print(f"\n--- 토큰화 테스트 ---")
print(f"영어 원문: {test_en}")
print(f"  토큰: {sp_en.encode(test_en, out_type=str)}")
print(f"  정수: {sp_en.encode(test_en, out_type=int)}")

print(f"\n한국어 원문: {test_ko}")
print(f"  토큰: {sp_ko.encode(test_ko, out_type=str)}")
print(f"  정수: {sp_ko.encode(test_ko, out_type=int)}")

# 특수 토큰 확인
print(f"\n--- 특수 토큰 ---")
print(f"PAD id: {sp_en.pad_id()} → '{sp_en.id_to_piece(sp_en.pad_id())}'")
print(f"UNK id: {sp_en.unk_id()} → '{sp_en.id_to_piece(sp_en.unk_id())}'")
print(f"BOS id: {sp_en.bos_id()} → '{sp_en.id_to_piece(sp_en.bos_id())}'")
print(f"EOS id: {sp_en.eos_id()} → '{sp_en.id_to_piece(sp_en.eos_id())}'")

영어 어휘 크기: 16000
한국어 어휘 크기: 16000

--- 토큰화 테스트 ---
영어 원문: Have you had dinner?
  토큰: ['▁Have', '▁you', '▁had', '▁dinner', '?']
  정수: [3691, 48, 245, 3750, 15928]

한국어 원문: 저녁은 드셨나요?
  토큰: ['▁저녁', '은', '▁드', '셨나요', '?']
  정수: [2804, 14688, 402, 6178, 14763]

--- 특수 토큰 ---
PAD id: 0 → '<pad>'
UNK id: 1 → '<unk>'
BOS id: 2 → '<s>'
EOS id: 3 → '</s>'


---
# 3. 데이터 전처리 (SentencePiece 기반)

### 기존 전처리 파이프라인 vs 새로운 파이프라인
```
[기존]
원문 → regex 필터 → Mecab morphs → 수동 vocab 구축 → 정수 인코딩 → 패딩

[새로운 - SentencePiece]
원문 → 소문자 변환(영어) → SentencePiece encode → 패딩
(vocab 구축, 정수 인코딩이 SentencePiece 내부에서 자동 처리)
```


In [22]:
# ============================================================
# SentencePiece 기반 데이터 전처리
# - 기존의 복잡한 regex + Mecab 파이프라인을 SentencePiece로 대체
# - <sos>, <eos>는 SentencePiece의 bos_id, eos_id 사용
# ============================================================

def preprocess_and_encode():
    encoder_input = []    # 영어 (소스)
    decoder_input = []    # 한국어 + <sos> (디코더 입력)
    decoder_target = []   # 한국어 + <eos> (디코더 레이블)

    for i in tqdm(range(num_samples), desc="전처리 중"):
        eng = en_sentences[i].lower().strip()  # 영어는 소문자 변환만
        kor = ko_sentences[i].strip()

        # SentencePiece 인코딩 (정수 시퀀스)
        enc_ids = sp_en.encode(eng, out_type=int)
        dec_ids = sp_ko.encode(kor, out_type=int)

        # 디코더 입력: <sos> + 토큰
        dec_input_ids = [sp_ko.bos_id()] + dec_ids
        # 디코더 레이블: 토큰 + <eos>
        dec_target_ids = dec_ids + [sp_ko.eos_id()]

        encoder_input.append(enc_ids)
        decoder_input.append(dec_input_ids)
        decoder_target.append(dec_target_ids)

    return encoder_input, decoder_input, decoder_target

encoder_input, decoder_input, decoder_target = preprocess_and_encode()

# 길이 통계
enc_lens = [len(s) for s in encoder_input]
dec_lens = [len(s) for s in decoder_input]
print(f"\n인코더 입력 길이 - 평균: {np.mean(enc_lens):.1f}, 최대: {max(enc_lens)}, 중간: {np.median(enc_lens):.0f}")
print(f"디코더 입력 길이 - 평균: {np.mean(dec_lens):.1f}, 최대: {max(dec_lens)}, 중간: {np.median(dec_lens):.0f}")

전처리 중:   0%|          | 0/88000 [00:00<?, ?it/s]

전처리 중: 100%|██████████| 88000/88000 [00:11<00:00, 7953.56it/s]


인코더 입력 길이 - 평균: 22.9, 최대: 263, 중간: 19
디코더 입력 길이 - 평균: 20.8, 최대: 293, 중간: 17


In [23]:
# ============================================================
# 패딩 및 데이터 분할
# ============================================================
def pad_sequences(sentences, max_len=None):
    if max_len is None:
        max_len = max([len(s) for s in sentences])
    features = np.zeros((len(sentences), max_len), dtype=int)
    for i, s in enumerate(sentences):
        length = min(len(s), max_len)
        features[i, :length] = np.array(s[:length])
    return features

encoder_input = pad_sequences(encoder_input)
decoder_input = pad_sequences(decoder_input)
decoder_target = pad_sequences(decoder_target)

print(f"인코더 입력: {encoder_input.shape}")
print(f"디코더 입력: {decoder_input.shape}")
print(f"디코더 레이블: {decoder_target.shape}")

# 셔플
indices = np.arange(encoder_input.shape[0])
np.random.seed(42)
np.random.shuffle(indices)
encoder_input = encoder_input[indices]
decoder_input = decoder_input[indices]
decoder_target = decoder_target[indices]

# 분할 (90% 훈련, 10% 검증)
n_of_val = int(num_samples * 0.1)
encoder_input_train, encoder_input_test = encoder_input[:-n_of_val], encoder_input[-n_of_val:]
decoder_input_train, decoder_input_test = decoder_input[:-n_of_val], decoder_input[-n_of_val:]
decoder_target_train, decoder_target_test = decoder_target[:-n_of_val], decoder_target[-n_of_val:]

print(f"\n훈련: {encoder_input_train.shape[0]}개, 검증: {encoder_input_test.shape[0]}개")

인코더 입력: (88000, 263)
디코더 입력: (88000, 293)
디코더 레이블: (88000, 293)

훈련: 79200개, 검증: 8800개


In [24]:
# ============================================================
# 인코딩/디코딩 확인
# ============================================================
idx = 0
print("--- 샘플 확인 ---")
print(f"인코더 입력 (정수): {encoder_input_train[idx][:20]}...")

# 영어 복원
enc_ids = [int(x) for x in encoder_input_train[idx] if x != 0]
print(f"영어 복원: {sp_en.decode(enc_ids)}")

# 한국어 복원 (디코더 입력에서 <sos> 제거)
dec_ids = [int(x) for x in decoder_input_train[idx] if x != 0 and x != sp_ko.bos_id()]
print(f"한국어 복원: {sp_ko.decode(dec_ids)}")

--- 샘플 확인 ---
인코더 입력 (정수): [   57  1914  8656 15913  4079    37   200   190  7481 15951    26 12954
     0     0     0     0     0     0     0     0]...
영어 복원: it shows leverage, trended out from 1919 to 2009.
한국어 복원: 부채가 1919년부터 2009년까지의 추세를 벗어나고 있음을 보여주고 있습니다.


---
# 4. 모델 정의 (Transformer + Pre-LN)

### 최적화 사항
1. **SentencePiece BPE** → 어휘 크기 16K, OOV 해소
2. **Pre-Layer Normalization** → 학습 안정성 대폭 향상
3. **GELU 활성화** → ReLU 대비 부드러운 기울기
4. **Attention Dropout** → 어텐션 가중치 정규화
5. **Xavier 가중치 초기화** → 학습 초기 수렴 가속
6. **6층 인코더/디코더** → 표현력 강화
7. **Label Smoothing + AdamW + Warmup/Cosine LR + Gradient Clipping + Early Stopping**


In [25]:
# 하이퍼파라미터
d_model = 512
nhead = 8
num_encoder_layers = 6
num_decoder_layers = 6
d_ff = 2048
dropout = 0.2
attn_dropout = 0.1

In [26]:
# ============================================================
# Positional Encoding + Dropout
# ============================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])

In [27]:
# MultiHeadAttention + Attention Dropout
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head, attn_dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.d_model = d_model
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.fc_out = nn.Linear(d_model, d_model)
        self.scale = math.sqrt(self.d_head)
        self.attn_dropout = nn.Dropout(attn_dropout)

    def forward(self, query, key, value, mask=None):
        B = query.shape[0]
        Q = self.w_q(query).view(B, -1, self.n_head, self.d_head).transpose(1, 2)
        K = self.w_k(key).view(B, -1, self.n_head, self.d_head).transpose(1, 2)
        V = self.w_v(value).view(B, -1, self.n_head, self.d_head).transpose(1, 2)

        energy = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        if mask is not None:
            energy = energy.masked_fill(mask == 0, -1e10)
        attention = self.attn_dropout(torch.softmax(energy, dim=-1))
        x = torch.matmul(attention, V)
        x = x.transpose(1, 2).contiguous().view(B, -1, self.d_model)
        return self.fc_out(x)

In [28]:
# FeedForward (GELU)
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(self.gelu(self.fc1(x))))

In [29]:
# ============================================================
# Pre-LN Encoder Layer
# x = x + sublayer(LN(x))  (기존 Post-LN: x = LN(x + sublayer(x)))
# ============================================================
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout, attn_dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_head, attn_dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        _x = x
        x = self.norm1(x)
        x = self.self_attn(x, x, x, mask)
        x = _x + self.dropout(x)

        _x = x
        x = self.norm2(x)
        x = self.feed_forward(x)
        x = _x + self.dropout(x)
        return x

In [30]:
# Pre-LN Decoder Layer
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout, attn_dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_head, attn_dropout)
        self.enc_dec_attn = MultiHeadAttention(d_model, n_head, attn_dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, tar_mask):
        _x = x; x = self.norm1(x)
        x = self.self_attn(x, x, x, tar_mask)
        x = _x + self.dropout(x)

        _x = x; x = self.norm2(x)
        x = self.enc_dec_attn(x, enc_output, enc_output, src_mask)
        x = _x + self.dropout(x)

        _x = x; x = self.norm3(x)
        x = self.feed_forward(x)
        x = _x + self.dropout(x)
        return x

In [31]:
# ============================================================
# Transformer (Pre-LN + Xavier Init + Final LN)
# ============================================================
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tar_vocab_size, d_model, nhead,
                 num_enc_layers, num_dec_layers, d_ff, dropout, attn_dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=0)
        self.tar_embedding = nn.Embedding(tar_vocab_size, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        self.pos_decoder = PositionalEncoding(d_model, dropout=dropout)

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, nhead, d_ff, dropout, attn_dropout)
            for _ in range(num_enc_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, nhead, d_ff, dropout, attn_dropout)
            for _ in range(num_dec_layers)
        ])

        # Pre-LN에서는 최종 출력 전 LayerNorm 필수
        self.encoder_norm = nn.LayerNorm(d_model)
        self.decoder_norm = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, tar_vocab_size)
        self.src_pad_idx = 0
        self.tar_pad_idx = 0

        # Xavier 초기화
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def make_src_mask(self, src):
        return (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)

    def make_tar_mask(self, tar):
        tar_pad_mask = (tar != self.tar_pad_idx).unsqueeze(1).unsqueeze(2)
        tar_len = tar.shape[1]
        tril = torch.tril(torch.ones((tar_len, tar_len), device=tar.device)).bool()
        return tar_pad_mask & tril.unsqueeze(0).unsqueeze(0)

    def forward(self, src, tar):
        src_mask = self.make_src_mask(src)
        tar_mask = self.make_tar_mask(tar)

        src = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        tar = self.pos_decoder(self.tar_embedding(tar) * math.sqrt(self.d_model))

        enc_out = src
        for layer in self.encoder_layers:
            enc_out = layer(enc_out, src_mask)
        enc_out = self.encoder_norm(enc_out)

        dec_out = tar
        for layer in self.decoder_layers:
            dec_out = layer(dec_out, enc_out, src_mask, tar_mask)
        dec_out = self.decoder_norm(dec_out)

        return self.fc_out(dec_out)

model = Transformer(
    src_vocab_size, tar_vocab_size, d_model, nhead,
    num_encoder_layers, num_decoder_layers, d_ff, dropout, attn_dropout
)
model.to(device)

# 멀티 GPU 사용
if torch.cuda.device_count() > 1:
    print(f"DataParallel: {torch.cuda.device_count()}개 GPU 사용")
    model = nn.DataParallel(model, device_ids=[0, 1])

total_params = sum(p.numel() for p in model.parameters())
print(f"총 파라미터 수: {total_params:,}")
print(f"\n모델 구조:\n{model}")

DataParallel: 4개 GPU 사용
총 파라미터 수: 68,732,544

모델 구조:
DataParallel(
  (module): Transformer(
    (src_embedding): Embedding(16000, 512, padding_idx=0)
    (tar_embedding): Embedding(16000, 512, padding_idx=0)
    (pos_encoder): PositionalEncoding(
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (pos_decoder): PositionalEncoding(
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (encoder_layers): ModuleList(
      (0-5): 6 x EncoderLayer(
        (self_attn): MultiHeadAttention(
          (w_q): Linear(in_features=512, out_features=512, bias=True)
          (w_k): Linear(in_features=512, out_features=512, bias=True)
          (w_v): Linear(in_features=512, out_features=512, bias=True)
          (fc_out): Linear(in_features=512, out_features=512, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
        )
        (feed_forward): PositionwiseFeedForward(
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_featur

---
# 5. 학습 설정 및 훈련

In [32]:
# Label Smoothing Loss
class LabelSmoothingLoss(nn.Module):
    def __init__(self, vocab_size, padding_idx=0, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.vocab_size = vocab_size
        self.padding_idx = padding_idx
        self.criterion = nn.KLDivLoss(reduction='batchmean')

    def forward(self, pred, target):
        pred = pred.log_softmax(dim=-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.vocab_size - 2))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
            true_dist[:, self.padding_idx] = 0
            true_dist[target == self.padding_idx] = 0
        return self.criterion(pred, true_dist)

loss_function = LabelSmoothingLoss(tar_vocab_size, padding_idx=0, smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=5e-4, betas=(0.9, 0.98), eps=1e-9, weight_decay=1e-5)

num_epochs = 20
warmup_epochs = 3
scheduler = optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs),
        optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs - warmup_epochs, eta_min=1e-6)
    ],
    milestones=[warmup_epochs]
)

In [ ]:
# 데이터로더
batch_size = 128
train_dataset = TensorDataset(
    torch.tensor(encoder_input_train, dtype=torch.long),
    torch.tensor(decoder_input_train, dtype=torch.long),
    torch.tensor(decoder_target_train, dtype=torch.long)
)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

valid_dataset = TensorDataset(
    torch.tensor(encoder_input_test, dtype=torch.long),
    torch.tensor(decoder_input_test, dtype=torch.long),
    torch.tensor(decoder_target_test, dtype=torch.long)
)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
model.to(device)
print(f"훈련 배치: {len(train_dataloader)}, 검증 배치: {len(valid_dataloader)}")

In [34]:
def evaluation(model, dataloader, loss_function, device):
    model.eval()
    total_loss, total_correct, total_count = 0.0, 0.0, 0.0
    with torch.no_grad():
        for enc_in, dec_in, dec_tar in dataloader:
            enc_in, dec_in, dec_tar = enc_in.to(device), dec_in.to(device), dec_tar.to(device)
            output = model(enc_in, dec_in)
            loss = loss_function(output.view(-1, output.size(-1)), dec_tar.view(-1))
            total_loss += loss.item()
            mask = dec_tar != 0
            total_correct += ((output.argmax(dim=-1) == dec_tar) * mask).sum().item()
            total_count += mask.sum().item()
    return total_loss / len(dataloader), total_correct / total_count

In [35]:
# Training Loop
best_val_loss = float('inf')
patience = 5
patience_counter = 0
train_losses_log, valid_losses_log, valid_accs_log, lr_log = [], [], [], []

for epoch in range(num_epochs):
    model.train()
    for enc_in, dec_in, dec_tar in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        enc_in, dec_in, dec_tar = enc_in.to(device), dec_in.to(device), dec_tar.to(device)
        optimizer.zero_grad()
        outputs = model(enc_in, dec_in)
        loss = loss_function(outputs.view(-1, outputs.size(-1)), dec_tar.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    train_loss, train_acc = evaluation(model, train_dataloader, loss_function, device)
    valid_loss, valid_acc = evaluation(model, valid_dataloader, loss_function, device)

    train_losses_log.append(train_loss)
    valid_losses_log.append(valid_loss)
    valid_accs_log.append(valid_acc)
    lr_log.append(current_lr)

    print(f'Epoch {epoch+1}/{num_epochs} | LR: {current_lr:.6f}')
    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
    print(f'  Valid Loss: {valid_loss:.4f} | Valid Acc: {valid_acc:.4f}')

    if valid_loss < best_val_loss:
        print(f'  ✅ Improved: {best_val_loss:.4f} → {valid_loss:.4f}')
        best_val_loss = valid_loss
        patience_counter = 0
        # DataParallel 감싸진 경우 module의 state_dict 저장
        state_dict = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        torch.save(state_dict, 'transformer_v2_best.pth')
    else:
        patience_counter += 1
        print(f'  ⚠️ No improvement ({patience_counter}/{patience})')
        if patience_counter >= patience:
            print(f'\n🛑 Early Stopping at epoch {epoch+1}!')
            break

print(f'\n학습 완료! Best Valid Loss: {best_val_loss:.4f}')

Epoch 1/20:   0%|          | 0/310 [00:01<?, ?it/s]


OutOfMemoryError: Caught OutOfMemoryError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/home/ryu4090/anaconda3/envs/playground/lib/python3.11/site-packages/torch/nn/parallel/parallel_apply.py", line 103, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ryu4090/anaconda3/envs/playground/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ryu4090/anaconda3/envs/playground/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_2572448/2258133315.py", line 61, in forward
    dec_out = layer(dec_out, enc_out, src_mask, tar_mask)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ryu4090/anaconda3/envs/playground/lib/p

In [ ]:
# 학습 곡선
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(train_losses_log, label='Train', marker='o')
axes[0].plot(valid_losses_log, label='Valid', marker='s')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(valid_accs_log, label='Valid Acc', marker='o', color='green')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True)
axes[2].plot(lr_log, label='LR', marker='o', color='red')
axes[2].set_title('Learning Rate'); axes[2].legend(); axes[2].grid(True)
plt.tight_layout(); plt.savefig('transformer_v2_training.png', dpi=150); plt.show()

---
# 6. 추론 및 평가

In [ ]:
# DataParallel 내부의 원본 모델에 가중치 로드
base_model = model.module if hasattr(model, 'module') else model
base_model.load_state_dict(torch.load('transformer_v2_best.pth'))
val_loss, val_acc = evaluation(model, valid_dataloader, loss_function, device)
print(f'Best model - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')

In [ ]:
# ============================================================
# 추론 함수 (SentencePiece 기반)
# ============================================================
def decode_sequence(input_seq, model, max_output_len=150):
    model.eval()
    src = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(device)
    sos_id = sp_ko.bos_id()
    eos_id = sp_ko.eos_id()
    tar_input = torch.tensor([[sos_id]], dtype=torch.long).to(device)
    decoded_ids = []

    for _ in range(max_output_len):
        with torch.no_grad():
            output = model(src, tar_input)
        pred_id = output[:, -1, :].argmax(dim=-1).item()
        if pred_id == eos_id:
            break
        decoded_ids.append(pred_id)
        tar_input = torch.cat([tar_input, torch.tensor([[pred_id]], device=device)], dim=1)

    return sp_ko.decode(decoded_ids)

def ids_to_en(ids):
    return sp_en.decode([int(x) for x in ids if x != 0])

def ids_to_ko(ids):
    filtered = [int(x) for x in ids if x != 0 and x != sp_ko.bos_id() and x != sp_ko.eos_id()]
    return sp_ko.decode(filtered)

In [ ]:
# 번역 예시
print("=" * 60)
for seq_index in [1, 50, 100, 3050, 1000]:
    input_seq = encoder_input_train[seq_index]
    translated = decode_sequence(input_seq, model)
    print(f"입력: {ids_to_en(encoder_input_train[seq_index])}")
    print(f"정답: {ids_to_ko(decoder_input_train[seq_index])}")
    print(f"번역: {translated}")
    print("-" * 60)

In [ ]:
# BLEU / ROUGE 평가
import evaluate

bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")

def calculate_bleu_rouge(model, num_samples=1000):
    model.eval()
    predictions, references = [], []
    print(f"총 {num_samples}개 문장 평가 중...")
    for i in tqdm(range(num_samples)):
        target = ids_to_ko(decoder_target_test[i]).strip()
        predicted = decode_sequence(encoder_input_test[i], model).strip()
        if not predicted: predicted = " "
        predictions.append(predicted)
        references.append([target])

    bleu = bleu_metric.compute(predictions=predictions, references=references)
    rouge = rouge_metric.compute(predictions=predictions, references=references)

    print("\n" + "=" * 50)
    print(f"번역 성능 평가 ({num_samples}개)")
    print("=" * 50)
    print(f"BLEU  : {bleu['score']:.2f}")
    print(f"ROUGE-1: {rouge['rouge1']:.4f}")
    print(f"ROUGE-2: {rouge['rouge2']:.4f}")
    print(f"ROUGE-L: {rouge['rougeL']:.4f}")
    print("=" * 50)
    print(f"\n예시 - 정답: {references[0][0]}")
    print(f"       번역: {predictions[0]}")

calculate_bleu_rouge(model, num_samples=1000)